# 반도체 R&D 기술동향분석 AI Service

## 0. 환경 설정 및 라이브러리 Import

In [1]:
# ---------------------------------------------------------
# (패키지 설치는 터미널에서 'uv sync'로 자동 설치 권장)
# 수동 설치 시: !uv pip install langchain langgraph langchain-openai
#              tavily-python python-dotenv arxiv google-search-results
#              reportlab langchain-huggingface chromadb
# ---------------------------------------------------------

# [Cell 1] 모든 라이브러리 통합 Import
import os
from typing import TypedDict, Annotated
from dotenv import load_dotenv

# 1. LangGraph & LangChain Core
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages

# 2. 모델 및 도구 (LLM, Search, VectorDB)
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from serpapi import GoogleSearch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 3. 포맷팅 및 리포팅
# reportlab은 formatting_node 내부에서 직접 import (무거운 모듈이므로 필요 시점에 로드)
# pdfplumber는 data.ipynb에서 PDF 추출 용도로 사용 (agent.ipynb에서는 불필요)

# 4. 환경 변수 로드
load_dotenv()

print("환경 설정 완료")


환경 설정 완료


## 1. State 정의

In [2]:
class AgentState(TypedDict):
    # 1. 대화 기록
    messages: Annotated[list[BaseMessage], add_messages] 
    
    # 2. 오케스트레이션 상태
    next_node: str         
    
    # 3. 순차 데이터 파이프라인
    web_signal: str        # [Step 1] Web Agent가 찾아온 시장 동향/징후 데이터
    rag_context: str       # [Step 2] RAG Agent가 web_signal 기반으로 찾은 논문/팩트
    
    # 4. 결과물 상태
    analysis_result: str   # [Step 3] Tech Analyst가 판단한 위협 수준/TRL 데이터
    final_report: str      # [Step 4] Writer가 쓴 마크다운 초안
    
    # 5. 검색 성능 측정값
    retrieval_metrics: dict  # RAG 검색 성능 지표 (예: 검색된 문서 수, 유사도 점수 등)

## 2. Web Search Agent (시장 조사원)
Tavily API를 사용하되, 반드시 Pros(긍정) / Cons(부정) 다면적 검색(Adversarial Search)을 수행하여 할루시네이션 방지

In [3]:
def web_search_agent(state: AgentState):
    print("[1단계: Web Search] 시장 동향 및 특허 징후 탐색 중...")
    
    user_query = state["messages"][-1].content
    
    # 1. LLM을 통한 쿼리 최적화
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    query_gen_prompt = f"반도체 기술 분석을 위해 '{user_query}'에 대한 1)최신 뉴스 검색어와 2)구글 특허 검색어를 각각 하나씩만 뽑아줘. 형식: 뉴스:키워드, 특허:키워드"
    keywords = llm.invoke(query_gen_prompt).content
    
    # 2. Tavily 뉴스 검색
    tavily = TavilySearchResults(k=3)
    news_data = tavily.run(user_query) # 실제론 키워드 파싱 후 사용 권장
    
    # 3. SerpApi 특허 검색
    search = GoogleSearch({
        "engine": "google_patents",
        "q": user_query,
        "api_key": os.environ.get("SERPAPI_KEY"),
        "num": 3
    })
    patent_data = search.get_dict().get("organic_results", [])
    
    # 4. 결과 취합 (web_signal 생성)
    summary = f"--- 뉴스 동향 ---\n{str(news_data)[:500]}\n"
    summary += f"\n--- 특허 동향 ---\n"
    for p in patent_data:
        summary += f"- {p.get('title')}: {p.get('snippet')}\n"

    print("Web Search 완료. 'web_signal'에 저장되었습니다.")
    
    # 다음 목적지를 명시하고 상태 업데이트
    return {
        "web_signal": summary,
        "next_node": "supervisor"
    }

## 3. RAG Agent (연구원)

In [4]:
def rag_agent(state: AgentState):
    print("🔬 [2단계: RAG] 로컬 지식 베이스 분석 및 검색 성능 측정 중...")
    
    user_query = state["messages"][-1].content
    web_signal = state.get("web_signal", "")
    
    # 1. 로컬 임베딩 모델 로드
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    # 2. 벡터 DB 연결
    vectorstore = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings
    )
    
    # 3. 힌트 기반 쿼리 생성
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    rag_query_prompt = f"""
    사용자 질문: {user_query}
    최신 시장 징후: {web_signal[:500]}
    위 정보를 바탕으로 기술적 근거를 찾기 위한 심층 검색어 2개만 만들어줘.
    """
    refined_query = llm.invoke(rag_query_prompt).content
    print(f"   [Debug] 💡 RAG 검색어: {refined_query}") # 디버깅용 출력
    
    # 4. 심층 검색 수행 (K=4)
    K = 4
    retriever = vectorstore.as_retriever(search_kwargs={"k": K})
    relevant_docs = retriever.invoke(refined_query)
    
    # ---------------------------------------------------------
    #  Hit Rate@K 및 MRR 측정 로직
    # ---------------------------------------------------------
    hit_count = 0
    reciprocal_rank = 0
    
    for i, doc in enumerate(relevant_docs):
        # 300자에서 800자로 확장하여 본문까지 충분히 읽도록 수정
        eval_prompt = f"""
        질문: {user_query}
        문서 내용: {doc.page_content[:800]} 
        위 문서가 질문에 답하는 데 조금이라도 기술적 힌트나 도움이 되나요? 
        반드시 'YES' 또는 'NO'로 시작하고, 그 이유를 짧게 한 줄로 설명해.
        """
        response = llm.invoke(eval_prompt).content.strip()
        print(f"   [Debug] 문서 {i+1} 평가: {response}") # LLM의 채점 이유 출력
        
        if response.lower().startswith('yes'):
            if hit_count == 0: 
                hit_count = 1
                reciprocal_rank = 1 / (i + 1)
    
    metrics = {
        "hit_rate": float(hit_count),
        "mrr": float(reciprocal_rank),
        "k": K
    }
    # ---------------------------------------------------------

    # 5. 검색 결과 정리
    context_text = "--- 로컬 지식 베이스 검색 결과 ---\n"
    for i, doc in enumerate(relevant_docs):
        source = doc.metadata.get("source_id", "Unknown")
        context_text += f"[{i+1}] 출처: {source}\n내용: {doc.page_content}\n\n"

    print(f" RAG 완료 (Hit@{K}: {metrics['hit_rate']}, MRR: {metrics['mrr']:.2f})")
    
    return {
        "rag_context": context_text,
        "retrieval_metrics": metrics,
        "next_node": "supervisor"
    }

## 4. Tech Analyst Agent (수석 분석가)
rag_data와 web_data로 수집한 정보들을 분석해 'TRL 기반 간접 추정' 및 '위협 수준(High/Med/Low)' 평가

In [ ]:
def tech_analyst_agent(state: AgentState):
    print("[3단계: Analyst] 수집 데이터 통합 분석 및 위협 수준 평가 중...")
    
    user_query = state["messages"][-1].content
    web_signal = state.get("web_signal", "")
    rag_context = state.get("rag_context", "")
    
    llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    
    analysis_prompt = f"""
    당신은 SK Hynix R&D 전략실의 '수석 기술 전략 분석가'입니다.
    아래 수집된 데이터를 바탕으로 '{user_query}'에 대한 심층 경쟁 분석을 수행하세요.

    [데이터 1: 시장 징후 및 뉴스]
    {web_signal}

    [데이터 2: 기술 원리 및 논문/특허 팩트]
    {rag_context}

    ---
    ## ★ 경쟁사 선정 원칙
    분석 대상 기술 도메인에 따라 아래 목록에서 실질적으로 경쟁 관계에 있는 기업만 선정하세요.
    HBM/DRAM에 한정하지 말고, 쿼리의 기술 도메인을 먼저 파악한 뒤 해당 기업들을 포함하세요.

    [도메인별 주요 경쟁사 목록]
    - 메모리(HBM/DRAM/NAND)       : Samsung, Micron, Kioxia, WD
    - Logic 파운드리(미세공정/GAA)  : TSMC, Intel Foundry, Samsung Foundry, Rapidus, GlobalFoundries
    - 첨단 패키징(CoWoS/Foveros)   : TSMC, Intel, ASE Group, Amkor, SPIL
    - 전력반도체(GaN/SiC)           : Infineon, STMicroelectronics, ON Semiconductor,
                                     Wolfspeed, ROHM, Mitsubishi Electric, Renesas
    - AI 가속기/NPU                 : NVIDIA, AMD, Intel(Gaudi), Google(TPU), Apple, Qualcomm, Arm
    - 장비(EUV/ALD/CVD)             : ASML, Applied Materials, Lam Research, Tokyo Electron(TEL), KLA
    - 소재(유리기판/High-k/페로브스카이트)   : AGC, Corning, Samsung Electro-Mechanics, Merck, Air Products
    - 포토닉스/광인터커넥트          : Intel, Broadcom, Ayar Labs, Lightmatter, Marvell
    - 뉴로모픽/저항메모리(RRAM/ReRAM): IBM, Intel(Loihi), BrainScaleS, CrossBar, Weebit Nano
    → 쿼리 내용에서 기술 도메인을 판단하여 위 목록 중 관련 경쟁사를 3~6개 선정하세요.
      단, SK Hynix와 기술 영역이 겹치는 곳이라면 반드시 포함하세요.

    ---
    ## 분석 가이드라인

    ### A. TRL 기반 기술 성숙도 추정
    선정된 경쟁사 각각에 대해 TRL 단계를 추정하세요.
    TRL 기준:
      - TRL 1~3: 기초 연구·개념 검증 (논문·학회 발표가 주 근거, 비교적 공개됨)
      - TRL 4~6: 부품/시스템 개발 (영업비밀 구간 → 간접 지표로 추정 필수)
      - TRL 7~9: 시제품~실제 양산 (고객 샘플·양산 발표·실적 공시로 확인)

    ⚠️ TRL 4~6 구간은 반드시 아래 '간접 지표'를 활용했음을 명시하세요:
      - 특허 출원 패턴 (출원 키워드·빈도·CPC 분류 변화)
      - 주요 학회(IEDM, ISSCC, Hot Chips, VLSI, ECTC 등) 발표 빈도/주제 변화
      - 경쟁사 채용 공고 키워드 징후
      - 파트너십·공급망·고객사 계약 발표 패턴

    ### B. 경쟁사별 전략 비교
    선정된 각 경쟁사에 대해:
      - 핵심 기술 전략 방향
      - SK Hynix 대비 강점·약점·차별화 요소
      - '게임체인저'가 될 수 있는 기술 요소
      - 발표(IR/뉴스)와 실제 양산 능력 사이의 Gap

    ### C. 위협 수준 종합 평가
    경쟁사별·기술별 위협을 (Low / Medium / High / Critical) 4단계로 평가하고
    반드시 데이터 근거를 제시하세요.

    ### D. 전략적 시사점 (SK Hynix 관점)
    - 현재 우위를 유지하거나 확장하기 위한 핵심 R&D 액션
    - 단기(6~12개월) / 중기(1~2년) / 장기(3~5년) 시계로 구분
    ---

    응답은 전문적인 한국어로 작성하고, 추정에는 "~로 판단됨", "~할 가능성이 높음" 등 분석적 어조를 유지하세요.
    """
    
    analysis_response = llm.invoke(analysis_prompt).content
    print("분석 완료. 'analysis_result'에 종합 평가가 저장되었습니다.")
    
    return {
        "analysis_result": analysis_response,
        "next_node": "supervisor"
    }


## 6. Report Writer (편집자)

In [ ]:
def report_writer_agent(state: AgentState):
    print("[4단계: Writer] 기술 전략 분석 보고서 작성 중...")

    user_query      = state["messages"][-1].content
    analysis_result = state.get("analysis_result", "")
    rag_context     = state.get("rag_context", "")
    web_signal      = state.get("web_signal", "")
    metrics         = state.get("retrieval_metrics", {})

    llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

    hit_rate = metrics.get("hit_rate", "N/A")
    mrr_val  = metrics.get("mrr", 0)
    mrr_str  = f"{mrr_val:.2f}" if isinstance(mrr_val, float) else str(mrr_val)
    k_val    = metrics.get("k", "N/A")

    # ── Step 0: 보고서 제목 생성 ──────────────────────────────────
    title_prompt = f"""
다음 분석 쿼리로 '기술 전략 분석 보고서' 제목을 만들어줘.
쿼리: '{user_query}'

규칙:
- 질문 그대로 쓰지 말 것 / 요청형 표현 금지
- 핵심 기술 도메인 + 분석 관점을 담은 명사형 제목
- 예시: '차세대 HBM4 하이브리드 본딩 기술 동향: 경쟁사 TRL 비교 및 위협 평가'
- 한 줄, 따옴표 없이 출력
"""
    report_title = llm.invoke(title_prompt).content.strip().strip('"\'')
    print(f"  제목: {report_title}")

    # ── Step 1: 메인 보고서 ──────────────────────────────────────
    report_prompt = f"""
당신은 반도체 R&D 전략 보고서 전문 편집자입니다.
SK Hynix R&D 담당자가 즉시 활용할 수 있는 '기술 전략 분석 보고서'를 작성하세요.
목표 분량: 약 2,500~3,500자 (섹션별 최소 300자).

[보고서 제목]
{report_title}

[수집된 분석 결과 (Tech Analyst)]
{analysis_result}

[RAG 지식베이스 원문 발췌]
{rag_context[:3000] if rag_context else "(없음)"}

[웹 검색 수집 자료]
{web_signal[:2000] if web_signal else "(없음)"}

[RAG 검색 신뢰도]
- Hit Rate@{k_val}: {hit_rate}  /  MRR: {mrr_str}

---
아래 5개 섹션을 **반드시 이 순서·형식**으로 작성하세요.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# {report_title}

> **분석 기준일**: (오늘 날짜)  |  **분류**: 대외비 R&D 전략 자료  |  **작성**: AI 분석 시스템

---

## 1. Executive Summary

3~5문장의 핵심 결론 요약 (가장 중요한 위협·기회 포인트 포함).

**핵심 시사점 (Top 3)**
| # | 시사점 | SK Hynix 대응 방향 |
|---|--------|-------------------|
| 1 | ... | ... |
| 2 | ... | ... |
| 3 | ... | ... |

---

## 2. 핵심 기술 영역별 R&D 동향

### 2-1. 재료·소재
쿼리 도메인과 관련된 소재 기술 동향 (유리 기판, GaN/SiC, High-k 유전체 등).
수집된 논문·보고서 근거 명시 필수.

### 2-2. 공정·장비
미세화 한계 돌파 기술(EUV, High-NA EUV, ALD, GAA 등) 동향.
주요 장비 업체 및 기술 성숙도.

### 2-3. 아키텍처·패키징
HBM4 하이브리드 본딩, CXL, NPU/AI 가속기 패키징 동향.
플레이어별 기술 차별화 포인트.

---

## 3. 경쟁사 기술 성숙도 및 위협 수준 분석

### 3-1. TRL 기반 기술 성숙도 비교

> ⚠️ **분석 한계 고지**
> 이 보고서에서 제시하는 TRL 수치는 **직접 실험·검증이 불가능한 추정치**입니다.
> TRL 4~6 구간은 특히 불확실성이 높습니다.
> 아래 간접 지표를 근거로 추정하였으며, 실제 수치는 각 기업의 내부 상황에 따라 상이할 수 있습니다.
>
> **추정 근거로 사용된 간접 지표:**
> - 📄 특허 출원 패턴 (출원 건수 급증 여부, IPC 분류 변화)
> - 🎤 주요 학회 발표 빈도 변화 (IEDM, ISSCC, Hot Chips 등)
> - 💼 채용 공고 키워드 (LinkedIn, 각 사 채용 페이지의 기술 스택 변화)
> - 📰 산업 분석 보고서 및 공식 IR 발표 내용

| 기업 | 핵심 기술 | TRL 추정 | 추정 근거 (간접 지표) | 상용화 시점 | SK Hynix 대비 위치 |
|------|----------|---------|-------------------|------------|------------------|
| (예) TSMC | 2nm GAA | TRL 7~8 | IEDM 2024 발표·N2 고객사 테이프아웃 급증 | 2025 양산 | 파운드리 선두 |
| ... | ... | ... | ... | ... | ... |

(도메인별 3~5개 기업 선정: 메모리→Samsung/Micron/SK Hynix, 파운드리→TSMC/Intel Foundry/Samsung Foundry/Rapidus)

### 3-2. 위협 수준 평가

| 위협 항목 | 위협 수준 | 핵심 근거 | 모니터링 지표 |
|----------|---------|---------|------------|
| (예) Samsung HBM4E 조기 양산 | 🔴 High | 2024 IEDM 하이브리드 본딩 논문 3편·특허 급증 | 삼성 IEDM 발표·채용공고 |
| (예) Micron HBM4 후발 추격 | 🟡 Medium | HBM3E 수율 개선 중, HBM4 2026 목표 | Micron IR 발표, 특허 동향 |
| (예) 공급망 리스크 | 🟢 Low | 현재 TSMC 파운드리 의존도 낮음 | 장비 수입 데이터 |

---

## 4. R&D 전략적 대응 방안

### 4-1. 단기 확보 필수 기술 (0~2년)
즉시 투자·확보해야 할 기술 3~5개. 각 항목마다 근거 명시.

### 4-2. 중장기 선행 연구 타겟 (3~5년)
미래 경쟁력을 위한 선행 R&D 주제 3~5개.

### 4-3. 오픈 이노베이션 협력 포인트
- 산학연 협력 가능 기관 (국내: KAIST·POSTECH·ETRI, 해외: MIT·Imec 등)
- 스타트업 생태계 및 M&A 후보군
- 특허 크로스 라이센싱 전략

---

## 5. 분석 방법론 및 출처

> **이 보고서의 TRL 추정은 간접 지표 기반 추정임을 명시합니다.**
> 직접적인 기업 내부 데이터에 접근하지 않았으며,
> 공개된 학술자료·특허·보도자료·채용공고를 종합 분석하여 도출한 결과입니다.

**참고 문헌**
(수집된 RAG 문서 및 웹 자료를 아래 형식으로 나열)
1. [arXiv XXXX.XXXXX] 제목 (저자, 연도)
2. [보고서] ETRI/KISTEP 보고서명 (발행기관, 연도)
3. [Web] 출처명 - URL

**데이터 출처 신뢰도**
| 출처 유형 | 활용 데이터 | 신뢰도 |
|---------|-----------|-------|
| arXiv 논문 | 기술 트렌드·상세 구현 방법 | ★★★★☆ |
| 산업 보고서 (ETRI/KISTEP) | 시장 동향·국내 현황 | ★★★★★ |
| 웹 검색 (Tavily) | 최신 뉴스·공시 | ★★★☆☆ |
| TRL 추정값 | 간접 지표 기반 | ★★☆☆☆ (추정) |

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**보고서 작성 최종 주의사항**
- 제목은 '{report_title}' 그대로 사용
- TRL 추정치는 반드시 "TRL X~Y (추정)" 형식으로, Section 3 상단의 한계 고지 박스를 포함할 것
- 간접 지표(특허·학회·채용공고)를 근거로 명시하지 않은 TRL 수치 사용 금지
- 표는 마크다운 파이프(|) 형식 준수
- 분석 어조: '~로 판단됨', '~가능성이 높은 것으로 추정됨' 유지
- 시장 점유율 데이터가 있다면 Mermaid pie 차트 삽입 가능
"""
    report_content = llm.invoke(report_prompt).content
    print("보고서 작성 완료.")

    return {
        "final_report": report_content,
        "next_node": "supervisor"
    }


## 7. Formatting Node

In [7]:
def formatting_node(state: AgentState):
    print("[5단계: Formatting] 마크다운 보고서를 ReportLab으로 PDF 변환 중...")

    from datetime import datetime as _dt
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.units import mm
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        HRFlowable, PageBreak, KeepTogether
    )
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    import re

    report_md = state.get("final_report", "")
    if not report_md:
        print("변환할 보고서 내용이 없습니다.")
        return {"next_node": "supervisor"}

    # ── 폰트 등록 ────────────────────────────────────────────────────
    _dir = os.getcwd()
    _candidates = [
        os.path.join(_dir, 'NanumGothic.ttf'),
        os.path.join(_dir, 'font', 'NanumGothic.ttf'),
    ]
    _bold_candidates = [
        os.path.join(_dir, 'font', 'NanumGothicBold.ttf'),
        os.path.join(_dir, 'NanumGothicBold.ttf'),
    ]
    font_path      = next((p for p in _candidates      if os.path.exists(p)), _candidates[0])
    font_bold_path = next((p for p in _bold_candidates if os.path.exists(p)), font_path)

    if 'NG' not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont('NG',      font_path))
        pdfmetrics.registerFont(TTFont('NG-Bold', font_bold_path))
        pdfmetrics.registerFontFamily('NG', normal='NG', bold='NG-Bold')
    print(f'  폰트: {font_path}')

    # ── 색상 ─────────────────────────────────────────────────────────
    C_NAVY  = colors.HexColor('#1a365d')
    C_BLUE  = colors.HexColor('#2c5282')
    C_LGRAY = colors.HexColor('#f7fafc')
    C_GRAY  = colors.HexColor('#718096')

    # ── 스타일 ───────────────────────────────────────────────────────
    def S(name, **kw):
        base = dict(fontName='NG', fontSize=10, leading=16,
                    textColor=colors.black, spaceAfter=4)
        base.update(kw)
        return ParagraphStyle(name, **base)

    ST = {
        'title': S('title', fontName='NG-Bold', fontSize=17, textColor=C_NAVY,
                   spaceAfter=4, leading=24),
        'h2':    S('h2', fontName='NG-Bold', fontSize=13, textColor=C_NAVY,
                   spaceBefore=12, spaceAfter=5, leading=20),
        'h3':    S('h3', fontName='NG-Bold', fontSize=11, textColor=C_BLUE,
                   spaceBefore=8, spaceAfter=4, leading=17),
        'body':  S('body', fontSize=10, leading=17, spaceAfter=5),
        'bullet':S('bullet', fontSize=10, leading=17, leftIndent=14,
                   firstLineIndent=-8, spaceAfter=4),
        'warn':  S('warn', fontSize=9, leading=14, textColor=colors.HexColor('#c05621'),
                   leftIndent=8, spaceAfter=4),
        'cell':  S('cell', fontSize=9, leading=13),
        'cell_b':S('cell_b', fontName='NG-Bold', fontSize=9, leading=13),
        'footer':S('footer', fontSize=8, textColor=C_GRAY, leading=11),
        'meta':  S('meta', fontSize=9, textColor=C_GRAY, spaceAfter=8),
    }

    def cell_p(v, bold=False):
        if isinstance(v, Paragraph): return v
        return Paragraph(str(v), ST['cell_b' if bold else 'cell'])

    def make_table(data, col_widths):
        tdata = [[cell_p(c, i == 0) for c in row] for i, row in enumerate(data)]
        style = [
            ('BACKGROUND', (0,0), (-1,0), C_NAVY),
            ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
            ('GRID',       (0,0), (-1,-1), 0.4, colors.HexColor('#cbd5e0')),
            ('FONTSIZE',   (0,0), (-1,-1), 9),
            ('LEADING',    (0,0), (-1,-1), 13),
            ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
            ('TOPPADDING', (0,0), (-1,-1), 5),
            ('BOTTOMPADDING', (0,0), (-1,-1), 5),
            ('LEFTPADDING',   (0,0), (-1,-1), 6),
        ]
        for i in range(1, len(data)):
            if i % 2 == 0:
                style.append(('BACKGROUND', (0,i), (-1,i), C_LGRAY))
        t = Table(tdata, colWidths=col_widths)
        t.setStyle(TableStyle(style))
        return t

    # ── 마크다운 → ReportLab Flowable 변환 ──────────────────────────
    def md_to_flowables(md_text, page_w):
        """마크다운 텍스트를 ReportLab Flowable 리스트로 변환"""
        flowables = []
        lines = md_text.split('\n')
        i = 0

        def escape(t):
            """마크다운 인라인 서식 → ReportLab XML 태그 변환"""
            t = t.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            t = re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', t)
            t = re.sub(r'\*(.+?)\*',     r'<i>\1</i>', t)
            t = re.sub(r'`(.+?)`',        r'<font face="Courier">\1</font>', t)
            return t

        while i < len(lines):
            line = lines[i]

            # 빈 줄
            if not line.strip():
                flowables.append(Spacer(1, 4))
                i += 1
                continue

            # 마크다운 표 감지 (|로 시작하는 연속 줄)
            if line.strip().startswith('|'):
                table_lines = []
                while i < len(lines) and lines[i].strip().startswith('|'):
                    table_lines.append(lines[i])
                    i += 1
                # 구분선(---|---) 제거
                table_lines = [l for l in table_lines
                                if not re.match(r'^[\s|:\-]+$', l)]
                if not table_lines:
                    continue
                rows = []
                for tl in table_lines:
                    cells = [c.strip() for c in tl.strip().strip('|').split('|')]
                    rows.append(cells)
                if not rows:
                    continue
                n_cols = max(len(r) for r in rows)
                # 컬럼 폭 균등 분배
                col_w = [page_w / n_cols] * n_cols
                # 행 열 수 맞추기
                for r in rows:
                    while len(r) < n_cols:
                        r.append('')
                flowables.append(make_table(rows, col_w))
                flowables.append(Spacer(1, 8))
                continue

            # 수평선
            if re.match(r'^[-=_]{3,}$', line.strip()):
                flowables.append(HRFlowable(width='100%', thickness=0.5,
                                             color=C_BLUE, spaceAfter=6))
                i += 1
                continue

            # 제목
            m = re.match(r'^(#{1,3})\s+(.*)', line)
            if m:
                level = len(m.group(1))
                text  = escape(m.group(2))
                st = 'title' if level == 1 else ('h2' if level == 2 else 'h3')
                flowables.append(Paragraph(text, ST[st]))
                i += 1
                continue

            # 인용구 (>) → warn 스타일
            if line.strip().startswith('>'):
                text = escape(line.strip().lstrip('> '))
                flowables.append(Paragraph(text, ST['warn']))
                i += 1
                continue

            # 불릿 (- 또는 *)
            m = re.match(r'^(\s*)[-*]\s+(.*)', line)
            if m:
                indent = len(m.group(1))
                text   = escape(m.group(2))
                if indent >= 2:
                    p = Paragraph('  ·  ' + text, ST['bullet'])
                else:
                    p = Paragraph('•  ' + text, ST['bullet'])
                flowables.append(p)
                i += 1
                continue

            # 번호 목록
            m = re.match(r'^\s*\d+\.\s+(.*)', line)
            if m:
                flowables.append(Paragraph('•  ' + escape(m.group(1)), ST['bullet']))
                i += 1
                continue

            # 일반 문단
            flowables.append(Paragraph(escape(line.strip()), ST['body']))
            i += 1

        return flowables

    # ── PDF 생성 ─────────────────────────────────────────────────────
    _date = _dt.now().strftime('%Y%m%d')
    output_filename = f'./semiconductor_analysis_{_date}.pdf'
    W = A4[0] - 40 * mm  # 유효 본문 폭

    def add_page_number(canvas, doc):
        canvas.saveState()
        canvas.setFont('NG', 8)
        canvas.setFillColor(C_GRAY)
        canvas.drawRightString(A4[0] - 20*mm, 14*mm,
                               f'SK Hynix R&D 내부 참고용  |  p. {canvas.getPageNumber()}')
        canvas.drawString(20*mm, 14*mm, '⚠ 대외비 — 무단 배포 금지')
        canvas.restoreState()

    doc = SimpleDocTemplate(
        output_filename, pagesize=A4,
        leftMargin=20*mm, rightMargin=20*mm,
        topMargin=20*mm,  bottomMargin=22*mm,
    )

    story = md_to_flowables(report_md, W)

    try:
        doc.build(story, onFirstPage=add_page_number, onLaterPages=add_page_number)
        print(f'PDF 생성 완료! → {output_filename}')
    except Exception as e:
        print(f'시스템 오류: {e}')

    return {'next_node': 'end'}


## 8. Supervisor Agent 및 Workflow 연결 (오케스트레이터)

In [8]:
def supervisor_agent(state: AgentState):
    print("\n[Supervisor] 작업 현황 파악 및 라우팅 지시...")
    
    # 1. Web Search 먼저 (징후 포착)
    if not state.get("web_signal"):
        return {"next_node": "web_search_agent"}
    
    # 2. RAG 실행 (Web 결과를 바탕으로 기술 원리 탐색)
    elif not state.get("rag_context"):
        return {"next_node": "rag_agent"}
    
    # 3. 기술 분석 (두 데이터 통합)
    elif not state.get("analysis_result"):
        return {"next_node": "tech_analyst_agent"}
    
    # 4. 보고서 작성
    elif not state.get("final_report"):
        return {"next_node": "report_writer_agent"}
    
    # 5. 모든 데이터가 있고 보고서도 있다면 PDF 생성 후 종료
    else:
        return {"next_node": "formatting_node"}

# ---- LangGraph 조립 (Graph Compilation) ----
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("rag_agent", rag_agent)
workflow.add_node("web_search_agent", web_search_agent)
workflow.add_node("tech_analyst_agent", tech_analyst_agent)
workflow.add_node("report_writer_agent", report_writer_agent)
workflow.add_node("formatting_node", formatting_node)

# 엣지 연결 (Supervisor 중심의 Star 구조)
workflow.add_edge(START, "supervisor")
workflow.add_edge("rag_agent", "supervisor")
workflow.add_edge("web_search_agent", "supervisor")
workflow.add_edge("tech_analyst_agent", "supervisor")
workflow.add_edge("report_writer_agent", "supervisor")
workflow.add_edge("formatting_node", END)

# 조건부 엣지 (Supervisor의 결정에 따라 이동)
workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "web_search_agent": "web_search_agent",
        "rag_agent": "rag_agent",
        "tech_analyst_agent": "tech_analyst_agent",
        "report_writer_agent": "report_writer_agent",
        "formatting_node": "formatting_node",
        "end": END  # 종료 경로 명시
    }
)

app = workflow.compile()

## 9. 최종 실행

In [9]:
# ============================================================
# 분석 쿼리 예시 — 아래 중 하나를 골라 content에 넣고 실행하세요
# ============================================================

QUERY_EXAMPLES = {
    # ── 패키징 / 메모리 ───────────────────────────────────────────
    "HBM4_hybrid_bonding":
        "HBM4 세대에서 삼성·마이크론이 추진 중인 Hybrid Bonding 전략과"
        " SK Hynix 대비 위협 수준을 분석해줘",

    "HBM_thermal":
        "HBM 적층 단수 증가(12단→16단)에 따른 발열 문제 해결 기술 현황과"
        " 경쟁사별 열관리 전략을 분석해줘",

    "advanced_packaging":
        "CoWoS·Foveros·SoIC 등 첨단 패키징 기술의 TRL 비교와"
        " TSMC·Intel·ASE의 경쟁 위협 수준을 분석해줘",

    # ── 공정 기술 ─────────────────────────────────────────────────
    "2nm_GAA":
        "TSMC·Intel·삼성 파운드리의 2nm GAA 공정 기술 성숙도와"
        " BSPDN 도입 계획을 비교 분석해줘",

    "EUV_high_NA":
        "High-NA EUV 도입에 따른 공정 기술 변화와"
        " ASML 독점 공급 리스크, 국내 대응 현황을 분석해줘",

    # ── 소재 / 기판 ───────────────────────────────────────────────
    "glass_substrate":
        "유리 기판(Glass Substrate) 기술의 현재 TRL과"
        " Intel·AGC·삼성전기의 개발 동향 및 SK Hynix 패키징에 미치는 영향을 분석해줘",

    "high_k_metal_gate":
        "High-k/Metal Gate 소재 기술의 2nm 이하 공정 적용 한계와"
        " 차세대 대안 소재(HfO2 기반 페로전기체 등) 개발 현황을 분석해줘",

    # ── 전력반도체 ────────────────────────────────────────────────
    "GaN_SiC_power":
        "GaN·SiC 전력반도체 시장에서 Infineon·STMicro·Wolfspeed의"
        " TRL 비교와 국내 기업 대응 전략을 분석해줘",

    # ── 차세대 컴퓨팅 아키텍처 ────────────────────────────────────
    "PIM_CXL":
        "PIM(Processing-In-Memory)과 CXL 메모리 확장 기술의 상용화 수준을 비교하고"
        " SK Hynix AiMX·CXL 모듈의 경쟁 포지셔닝을 분석해줘",

    "RRAM_ReRAM_neuromorphic":
        "RRAM·ReRAM 기반 뉴로모픽 칩 개발 현황과"
        " IBM·Intel Loihi·CrossBar의 TRL 비교 및 NPU 대체 가능성을 분석해줘",

    "silicon_photonics":
        "Co-Packaged Optics(CPO) 및 실리콘 포토닉스 기술의 데이터센터 도입 현황과"
        " Intel·Broadcom·Ayar Labs의 위협 수준을 분석해줘",
}

# ▶ 실행할 쿼리를 여기서 선택하세요
SELECTED = "HBM4_hybrid_bonding"

inputs = {
    "messages": [HumanMessage(content=QUERY_EXAMPLES[SELECTED])]
}

print(f"🔍 분석 쿼리: {QUERY_EXAMPLES[SELECTED]}")
print("반도체 R&D 분석 파이프라인 가동...\n")

for output in app.stream(inputs, config={"recursion_limit": 20}):
    for key, value in output.items():
        print(f"\n--- Node '{key}' 실행 완료 ---")
        if "final_report" in value:
            print("✅ 보고서 작성 완료")


🔍 분석 쿼리: HBM4 세대에서 삼성·마이크론이 추진 중인 Hybrid Bonding 전략과 SK Hynix 대비 위협 수준을 분석해줘
반도체 R&D 분석 파이프라인 가동...


[Supervisor] 작업 현황 파악 및 라우팅 지시...

--- Node 'supervisor' 실행 완료 ---
[1단계: Web Search] 시장 동향 및 특허 징후 탐색 중...


/var/folders/3h/8skmr2d95fx0s1wb8f1l9m_40000gn/T/ipykernel_12025/1932448126.py:12: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(k=3)


Web Search 완료. 'web_signal'에 저장되었습니다.

--- Node 'web_search_agent' 실행 완료 ---

[Supervisor] 작업 현황 파악 및 라우팅 지시...

--- Node 'supervisor' 실행 완료 ---
🔬 [2단계: RAG] 로컬 지식 베이스 분석 및 검색 성능 측정 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/var/folders/3h/8skmr2d95fx0s1wb8f1l9m_40000gn/T/ipykernel_12025/1809200508.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


   [Debug] 💡 RAG 검색어: 1. "HBM4 Hybrid Bonding 기술 삼성 마이크론 비교 분석"
2. "SK Hynix HBM 기술 로드맵 2026 시장 경쟁력"
 RAG 완료 (Hit@4: 0.0, MRR: 0.00)

--- Node 'rag_agent' 실행 완료 ---

[Supervisor] 작업 현황 파악 및 라우팅 지시...

--- Node 'supervisor' 실행 완료 ---
[3단계: Analyst] 수집 데이터 통합 분석 및 위협 수준 평가 중...
분석 완료. 'analysis_result'에 종합 평가가 저장되었습니다.

--- Node 'tech_analyst_agent' 실행 완료 ---

[Supervisor] 작업 현황 파악 및 라우팅 지시...

--- Node 'supervisor' 실행 완료 ---
[4단계: Writer] 기술 전략 분석 보고서 작성 중...
  제목: HBM4 하이브리드 본딩 전략 분석: 삼성·마이크론 vs SK Hynix 경쟁력 평가
보고서 작성 완료.

--- Node 'report_writer_agent' 실행 완료 ---
✅ 보고서 작성 완료

[Supervisor] 작업 현황 파악 및 라우팅 지시...

--- Node 'supervisor' 실행 완료 ---
[5단계: Formatting] 마크다운 보고서를 ReportLab으로 PDF 변환 중...
  폰트: /Users/gani/Desktop/ai_semi/NanumGothic.ttf
PDF 생성 완료! → ./semiconductor_analysis_20260410.pdf

--- Node 'formatting_node' 실행 완료 ---
